# UMI in MuJoCo - Phase 1: Environment setup (Ubuntu / local GPU version)

Run this in Jupyter (JupyterLab or VS Code) on your Ubuntu machine. Run the cells in order. First run takes about 8-12 minutes (mostly the PyTorch download).

**Why a separate Python 3.9 environment?** The UMI repo pins `torch 2.1`, `numpy 1.24`, `diffusers 0.18.2`, `zarr 2.16` and others that do not install on a modern default Python. This notebook creates a Python 3.9 virtual environment at `~/umi_work/umi_env` and runs all UMI and MuJoCo code through it (`!{PY} script.py`). The Jupyter kernel itself is only used for notebook glue.

**Everything lives in `~/umi_work`** (same folder as Phase 0): the repo, the environment, your dataset, checkpoints and sim demos. Nothing is lost between sessions, so you only run this notebook once (re-run only if you delete the environment).

**Skipped on purpose (hardware-only):** `ur-rtde`, `spnav`, `v4l2py`, `pynput`, `atomics`, `fastcrc`, `PyExifTool`, `free-mujoco-py`, the old `robosuite` fork, and `ray` (only used for multi-run sweeps). We install the modern `mujoco` package instead.

## 1. Check the GPU and driver

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

**Note:** the pinned `torch 2.1.0` (CUDA 12.1 build) supports Turing, Ampere, Ada and Hopper GPUs (GTX 16xx/RTX 20/30/40, A100, H100, L4...) and needs NVIDIA driver 525 or newer. It does **not** support Blackwell GPUs (RTX 50xx, B100/B200). If you have one of those, tell me and I will change the pins.

## 2. System libraries (headless OpenGL for MuJoCo camera rendering, ffmpeg)
This needs `sudo`. If the cell cannot use sudo non-interactively, it prints the exact command to paste into a terminal; run that, then re-run this cell to confirm.

In [ ]:
%%bash
PKGS="libegl1 libgl1 libgl1-mesa-dri libglfw3 libosmesa6-dev patchelf ffmpeg"
MISSING=""
for p in $PKGS; do dpkg -s "$p" >/dev/null 2>&1 || MISSING="$MISSING $p"; done
if [ -z "$MISSING" ]; then echo "all system libraries already installed"; exit 0; fi
echo "missing:$MISSING"
if [ "$(id -u)" = "0" ]; then
  apt-get -qq update && apt-get -qq install -y $MISSING > /dev/null
elif sudo -n true 2>/dev/null; then
  sudo apt-get -qq update && sudo apt-get -qq install -y $MISSING > /dev/null
else
  echo ">> sudo needs a password. Run this in a terminal, then re-run this cell:"
  echo "   sudo apt-get update && sudo apt-get install -y$MISSING"
  exit 1
fi
echo "system libraries installed"

## 3. Paths
Everything goes under `~/umi_work` (the same `WORKDIR` as Phase 0). No Google Drive needed.

In [ ]:
import os, sys, shutil
from pathlib import Path

WORKDIR = Path.home() / 'umi_work'
ROOT = str(WORKDIR / 'umi_mujoco')                          # data, checkpoints, sim demos, logs
REPO = str(WORKDIR / 'universal_manipulation_interface')
ENV  = str(WORKDIR / 'umi_env')
PY   = f'{ENV}/bin/python'
KPY  = sys.executable                                       # the Jupyter kernel's Python
UV   = shutil.which('uv') or f'{KPY} -m uv'                 # installed in step 5 if missing

for sub in ['data', 'checkpoints', 'sim_demos', 'logs']:
    os.makedirs(f'{ROOT}/{sub}', exist_ok=True)

# make these visible to %%bash cells too
os.environ.update(WORKDIR=str(WORKDIR), ROOT=ROOT, REPO=REPO, ENV=ENV, PY=PY, KPY=KPY, UV=UV, WANDB_MODE='disabled')
print('WORKDIR:', WORKDIR)
print('ROOT   :', ROOT)
print('REPO   :', REPO)
print('PY     :', PY)

## 4. Clone the repo (skipped if it is already there from Phase 0)

In [ ]:
%%bash
if [ ! -d "$REPO/.git" ]; then
  git clone https://github.com/real-stanford/universal_manipulation_interface.git "$REPO"
fi
cd "$REPO" && git log -1 --format='repo at %h (%ad)'

### 4b. Link the dataset into the data folder
If Phase 0 downloaded `cup_in_the_wild.zarr.zip`, it is linked (not copied, so no extra disk space) into `~/umi_work/umi_mujoco/data/`.

In [ ]:
import os
URL   = 'https://real.stanford.edu/umi/data/zarr_datasets/cup_in_the_wild.zarr.zip'
DATA  = f'{ROOT}/data/cup_in_the_wild.zarr.zip'
candidates = [str(WORKDIR / 'cup_in_the_wild.zarr.zip'), f'{REPO}/cup_in_the_wild.zarr.zip']

if os.path.exists(DATA):
    print('dataset already in', DATA)
else:
    found = next((c for c in candidates if os.path.exists(c)), None)
    if found:
        os.symlink(found, DATA)
        print('linked', found, '->', DATA)
    else:
        print('dataset not found yet. Download it with:')
        print('  !wget -c -P', os.path.dirname(DATA), URL)

os.environ['DATA'] = DATA
print(DATA, '| exists:', os.path.exists(DATA), ('| %.2f GB' % (os.path.getsize(DATA) / 1e9)) if os.path.exists(DATA) else '')

## 5. Create the Python 3.9 environment
Uses `uv` (fast installer). If `uv` is not available it is pip-installed into the kernel's environment. If pip refuses because the system Python is "externally managed", run Jupyter from a conda env or venv instead, or install uv with `curl -LsSf https://astral.sh/uv/install.sh | sh` and restart Jupyter.

In [ ]:
%%bash
set -e
if ! command -v uv >/dev/null 2>&1 && ! "$KPY" -m uv --version >/dev/null 2>&1; then
  "$KPY" -m pip -q install uv || { echo ">> could not pip install uv (see note above)"; exit 1; }
fi
$UV python install 3.9
if [ ! -x "$PY" ]; then
  $UV venv "$ENV" --python 3.9 --seed
else
  echo "environment already exists at $ENV"
fi
"$PY" --version

## 6. Install dependencies (about 5-8 minutes)
Versions follow the repo's `conda_environment.yaml`, minus hardware-only packages, plus `mujoco`. `huggingface_hub` is pinned because `diffusers 0.18.2` breaks with newer versions. `mujoco` is pinned to 3.2.3.

In [ ]:
%%writefile ~/umi_work/umi_requirements.txt
torch==2.1.0
torchvision==0.16.0
numpy==1.24.4
scipy==1.11.4
numba==0.57.1
pandas==2.1.4
opencv-python-headless==4.7.0.72
matplotlib==3.7.5
av==10.0.0
pyyaml==6.0.1
tqdm==4.65.0
zarr==2.16.1
numcodecs==0.11.0
hydra-core==1.2.0
wandb==0.15.8
dill==0.3.7
einops==0.6.1
diffusers==0.18.2
huggingface_hub==0.19.4
scikit-video==1.1.11
lmdb
timm==0.9.7
scikit-fda==0.9
cython==0.29.37
accelerate==0.24.1
boto3
aiohttp==3.9.5
aiofiles==23.2.1
threadpoolctl==3.2.0
imagecodecs==2023.9.18
h5py
psutil
termcolor
tensorboardX
imageio
imageio-ffmpeg
mujoco==3.2.3

In [ ]:
%%bash
echo "requirements say:"; grep -E '^(av|mujoco)' ~/umi_work/umi_requirements.txt
set -eo pipefail
$UV pip install --python "$PY" -r ~/umi_work/umi_requirements.txt 2>&1 | tail -30
echo "core install finished"

### 6b. robomimic and gym
`robomimic` is imported by UMI's image encoder (`CropRandomizer`). It is installed without its own dependencies, which would pull in `egl_probe` and `robosuite`. `gym 0.21.0` needs old build tools, and is allowed to fail because training normally does not need it.

In [ ]:
%%bash
$UV pip install --python "$PY" --no-deps robomimic==0.2.0
"$PY" -m pip -q install "pip==23.3.2" "setuptools<66" "wheel<0.40"
"$PY" -m pip -q install --no-build-isolation gym==0.21.0 || echo ">> gym install failed (usually fine, continue)"

## 7. Verify: PyTorch, GPU and library versions

In [ ]:
%%bash
"$PY" - <<'EOF'
import sys, torch, numpy, zarr, numcodecs, imagecodecs, diffusers, timm, mujoco
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda build', torch.version.cuda, '| gpu available', torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    print('gpu', torch.cuda.get_device_name(0), '| %.1f GB VRAM' % (props.total_memory / 1e9), '| capability', cap)
    if cap[0] >= 10:
        print('WARNING: this GPU (Blackwell) is newer than torch 2.1 supports - tell Claude to change the pins')
    x = torch.randn(1024, 1024, device='cuda')
    print('gpu matmul ok:', bool(torch.isfinite(x @ x).all().item()))
else:
    print('WARNING: no GPU visible to torch (check `nvidia-smi` and that your driver is 525 or newer)')
print('numpy', numpy.__version__, '| zarr', zarr.__version__, '| numcodecs', numcodecs.__version__)
print('imagecodecs', imagecodecs.__version__, '| jpegxl available:', imagecodecs.JPEGXL.available)
print('diffusers', diffusers.__version__, '| timm', timm.__version__, '| mujoco', mujoco.__version__)
EOF

Expected: `gpu available True`, `numpy 1.24.4`, `zarr 2.16.1`, `jpegxl available: True`. If `jpegxl` is `False`, tell me.

## 8. Verify: every UMI module imports
This walks the `umi` and `diffusion_policy` packages and tries to import each module. Hardware-only modules are skipped or counted separately. Anything under **UNEXPECTED** is a real problem - paste it to me.

In [ ]:
%%bash
cd "$REPO"
export PYTHONPATH="$REPO"
"$PY" - <<'EOF'
import importlib, pkgutil
SKIP = ('real_world', 'env_runner', 'scripts')
HARDWARE = ('v4l2py', 'spnav', 'rtde', 'pynput', 'atomics', 'fastcrc', 'exiftool', 'robosuite', 'mujoco_py')
ok, expected, bad = [], [], []
for top in ('umi', 'diffusion_policy'):
    pkg = importlib.import_module(top)
    for m in pkgutil.walk_packages(pkg.__path__, top + '.', onerror=lambda n: None):
        name = m.name
        if any(s in name for s in SKIP):
            continue
        try:
            importlib.import_module(name)
            ok.append(name)
        except Exception as e:
            first = (str(e).splitlines() or [''])[0][:110]
            msg = type(e).__name__ + ': ' + first
            (expected if any(h in msg for h in HARDWARE) else bad).append((name, msg))
print('imported OK:', len(ok), 'modules')
print('failed only because of hardware/legacy packages (safe to ignore):', len(expected))
for n, msg in expected[:8]:
    print('   ', n, '->', msg)
print('UNEXPECTED failures:', len(bad))
for n, msg in bad:
    print('   ', n, '->', msg)
if bad:
    print('\nPaste the UNEXPECTED lines to Claude - these are the ones that matter.')
EOF

## 9. Verify: the training config resolves
Prints the UMI diffusion-policy config (no training). Check that `shape_meta` and `task.dataset_path` appear.

In [ ]:
%%bash
cd "$REPO"
export PYTHONPATH="$REPO"
"$PY" train.py --config-name=train_diffusion_unet_timm_umi_workspace --cfg job 2>&1 | head -150

## 10. Verify: MuJoCo offscreen rendering
Your simulated wrist camera will render exactly like this. It tries `egl` (GPU) first, then `osmesa` (CPU, slower but always works). Note which one prints a shape and use that backend in later phases. On an NVIDIA machine `egl` should work.

In [ ]:
%%writefile ~/umi_work/mujoco_render_test.py
import os, sys, numpy as np, mujoco, imageio

XML = '''
<mujoco>
  <visual><global offwidth="640" offheight="480"/></visual>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="table" type="plane" size="1 1 0.1" rgba="0.8 0.8 0.8 1"/>
    <body name="cube" pos="0 0 0.05">
      <freejoint/>
      <geom type="box" size="0.05 0.05 0.05" rgba="0.9 0.2 0.2 1"/>
    </body>
    <camera name="wrist" pos="0 -0.5 0.35" xyaxes="1 0 0 0 0.55 0.85" fovy="100"/>
  </worldbody>
</mujoco>
'''
model = mujoco.MjModel.from_xml_string(XML)
data = mujoco.MjData(model)
for _ in range(50):
    mujoco.mj_step(model, data)
renderer = mujoco.Renderer(model, height=224, width=224)
renderer.update_scene(data, camera='wrist')
img = renderer.render()
imageio.imwrite(os.path.expanduser('~/umi_work/mujoco_test.png'), img)
print('rendered', img.shape, img.dtype, '| pixel std = %.1f' % img.std())
sys.exit(0 if img.std() > 5 else 1)

In [ ]:
%%bash
for GL in egl osmesa; do
  echo "== MUJOCO_GL=$GL =="
  MUJOCO_GL=$GL PYOPENGL_PLATFORM=$GL "$PY" ~/umi_work/mujoco_render_test.py 2>&1 | tail -4
  if [ "${PIPESTATUS[0]}" = "0" ]; then echo ">> $GL works"; fi
done

In [ ]:
from IPython.display import Image, display
display(Image(str(WORKDIR / 'mujoco_test.png')))

## Phase 1 checklist

- [ ] Step 7: GPU visible, versions as expected
- [ ] Step 8: **UNEXPECTED failures: 0** (or pasted to Claude)
- [ ] Step 9: config printed, shows `shape_meta`
- [ ] Step 10: a red cube image appears; note the working GL backend (`egl` or `osmesa`)
- [ ] Dataset linked/downloaded (step 4b)

**For the next phases:** run scripts with `!{PY} your_script.py` (set `MUJOCO_GL=egl` or `osmesa` as found above) and add `PYTHONPATH=$REPO` when importing UMI code.

Next: Phase 2 (MuJoCo scene: arm, gripper, table, objects, wrist camera).